In [0]:
# Read all JSON files from Workspace path using Python native I/O
# (spark.read cannot access /Workspace/ paths on Serverless compute)
import json, glob
from pyspark.sql import DataFrame
from pyspark.sql.types import ArrayType, StructType, MapType
json_files = glob.glob("/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/New Extraction/folder_owner/*.json")

records = []
for f in json_files:
    with open(f) as fh:
        data = json.load(fh)
        # print()
        if isinstance(data.get('entries', []), list):
            records.extend(data.get('entries', []))
        else:
            records.append(data.get('entries', []))

print(f"Found {len(records)} records")

df = spark.createDataFrame(records)
from pyspark.sql import functions as sf

display(
    df.groupBy("SI_NAME").agg(
        sf.array_join(sf.collect_set("SI_OWNER"), " | ").alias("owners_agg"),
        sf.count("SI_ID").alias("si_id_count")
    )
)

In [0]:
%sql
select universe_name,record_type, Name, Description,Select, Where_expression,
* from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_idt_definitions
where record_type='Measure'

In [0]:
%sql
drop table dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.dev_sapbo_universe_tables

In [0]:
%sql
select distinct cms1.Document_Id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as aud1
on trim(cms1.Document_Id) = trim(aud1.WEBI_Doc_ID)
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_full as us1
on upper(trim(us1.uid))=upper(trim(aud1.UserName))
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_full as us2
on upper(trim(us2.uid))=upper(trim(coalesce(cms1.lastAuthor, cms1.created)))
where aud1.WEBI_Doc_ID is not null
and cms1.Document_Id not in (
    select distinct document_id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
)
and 
(
upper(Trim(us1.businessUnit)) in (upper('FINPM-FINANCE PROGRAM MANAGEMENT'),upper('GFO-GROUP FINANCE OPERATIONS'),upper('GFC-GROUP FINANCE AND CONTROL'),upper('COFIN-CORPORATE FINANCE & TAX'),upper('Group Finance'),upper('Finance'),upper('Finance & Control')) 
or 
us2.businessUnit in ('FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL','COFIN-CORPORATE FINANCE & TAX','Group Finance','Finance','Finance & Control')
)

select count(*) from _sqldf

In [0]:
%sql
select distinct cms1.Document_Id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms1
where cms1.Document_Id in (
    select distinct document_id from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
)
and cms1.DataSource_Type ='fhsql'
and cms1.SQL_Query in ('Error retrieving Query Plan')

In [0]:
%sql
select DataItem_name, universe_name, DataItem_description
from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
where lower(universe_name) like '%claim%' 


In [0]:
%sql
select count(distinct Document_Id ) from dataplatform01_central_dev_catalog_01.custom_sap_bo.pbi_webi_profile